# B4.1 — RAG: image top-5, cross-encoder, top-15 paragraphs

Same as B4 (top-5 + cross-encoder rerank) but keeps **15** paragraphs instead of 3. Tests whether feeding more (still reranked) context helps.

Accuracy appears once `results_B4_1.json` exists (eval).

In [ ]:
import json, os, re, string
OUT = '../../outputs'
BASE = '/work/cvcs2026/encyclopedic'
preds = [json.loads(l) for l in open(OUT + '/predictions_B4_1.jsonl') if l.strip()]
for label, f in [('A', 'results.json'), ('B4 (top3)', 'results_B4.json'),
                 ('B4.1 (top15)', 'results_B4_1.json')]:
    p = OUT + '/' + f
    if os.path.exists(p):
        print('%-14s overall %.4f' % (label, json.load(open(p))['accuracy_overall']))
    else:
        print('%-14s (pending)' % label)

## Where is the bottleneck? Answer accuracy on retrieval hit vs miss
Split by whether the correct article is among the top-5 candidates. Correctness is an exact-match proxy (floor of BEM); the hit-vs-miss gap is the signal.

In [ ]:
import matplotlib.pyplot as plt
PUNCT = set(string.punctuation + '\u2018\u2019\u00b4`_')

def _pre(a):
    a = a.lower().replace('\n', ' ').replace('\t', ' ').strip()
    a = ''.join('' if c in PUNCT else c for c in a)
    a = re.sub(r'\b(the answer is|a|an|the)\b', ' ', a)
    return ' '.join(a.split())

def correct(rec):
    if rec.get('prediction') is None:
        return False
    ref = str(rec['answer'])
    if rec['question_type'] == 'multi_answer':
        R = [x for x in (_pre(a) for a in ref.split('|')[0].split('&&')) if x]
        C = [x for x in (_pre(a) for a in rec['prediction'].replace(' and ', ',').replace(' & ', ',').split(',')) if x]
        u = len(set(R) | set(C))
        return len(set(R) & set(C)) / u >= 0.5 if u else False
    return _pre(ref) == _pre(rec['prediction'])

def norm(u):
    if not u:
        return u
    u = re.sub(r'^https?://', '', u.strip().lower())
    return u.replace('en.m.wikipedia.org', 'en.wikipedia.org').rstrip('/')

gt = {x['unique_id']: norm(x.get('wikipedia_url'))
      for x in json.load(open(BASE + '/encyclopedic_test_subset.json'))}

def cands(p):
    rc = p.get('retrieved_context') or {}
    cs = [c['wiki_url'] for c in rc.get('candidates', [])] or ([rc['wiki_url']] if rc.get('wiki_url') else [])
    return [norm(u) for u in cs]

hit = [p for p in preds if gt.get(p['unique_id']) in cands(p)]
miss = [p for p in preds if gt.get(p['unique_id']) not in cands(p)]
acc = lambda s: sum(correct(p) for p in s) / len(s) if s else 0
print('recall (article retrieved): %.1f%%' % (100 * len(hit) / len(preds)))
print('accuracy | HIT : %.3f (n=%d)' % (acc(hit), len(hit)))
print('accuracy | MISS: %.3f (n=%d)' % (acc(miss), len(miss)))
plt.figure(figsize=(4, 3.5))
plt.bar(['hit', 'miss'], [acc(hit), acc(miss)])
plt.ylabel('answer accuracy (EM proxy)'); plt.title('B4.1 bottleneck: retrieval')
plt.show()

## Pros / cons

**Pros**
- More reranked context than B4 (top-3) while still filtered by the cross-encoder.

**Cons / to check against B4**
- Longer prompts; if accuracy is ~equal to B4 (top-3), the extra paragraphs add little — the first few reranked paragraphs already carry the signal.
- Same retrieval ceiling: gains are bounded by recall@5, not by how many paragraphs are kept.